# Experimentos do Benchmark

### Bibliotecas & Algoritmos

In [2]:
import csv
import itertools
import multiprocessing as mp

from dataclasses import asdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from utils import gerador, medicao

### Configurações do Benchmark

In [3]:
TIMEOUT_S = 45.0
REPETICOES = 3
SEMENTE_BASE = 67

TAMANHOS = [10, 15, 20, 25, 30, 35, 40, 45, 50]
TIPOS = ["com_solucao", "sem_solucao", "alvo_pequeno", "alvo_grande"]

# Limita o paralelismo aos núcleos de desempenho (P-cores) para que nenhuma
# medição rode nos núcleos de eficiência e distorça a comparação.
MAX_WORKERS = medicao.nucleos_desempenho()

RESULTADOS_BACKTRACKING_VS_MIM_PATH = "resultados/medicoes.csv"
RESULTADOS_BACKTRACKING_EFICIENCIA_PODA = "resultados/medicoes_poda.csv"

### Funções Auxiliares

In [4]:
CAMPOS_CSV = [
    "algoritmo",
    "tipo_instancia",
    "n",
    "repeticao",
    "tempo_s",
    "pico_memoria_bytes",
    "estados",
    "existe",
    "estourou_timeout",
]

def salvar_csv(medicoes, caminho: str):
    Path(caminho).parent.mkdir(parents=True, exist_ok=True)
    with open(caminho, "w", newline="", encoding="utf-8") as arq:
        escritor = csv.DictWriter(arq, fieldnames=CAMPOS_CSV)
        escritor.writeheader()
        for resultado in medicoes:
            escritor.writerow(asdict(resultado))

## Instâncias

#### Executor de instancias

In [5]:
CENARIOS = list(itertools.product(TIPOS, TAMANHOS))

def medir(tipo, tamanho, algoritmo, repeticao, instancia, usar_poda):
    resultado = medicao.medir_execucao(algoritmo, instancia.transacoes, instancia.alvo, TIMEOUT_S, usar_poda)

    if algoritmo == "backtracking" and not usar_poda:
      algoritmo = "backtracking_sem_poda"

    if resultado is None:
        return medicao.Medicao(algoritmo, tipo, tamanho, repeticao, None, None, None, None, True)

    return medicao.Medicao(
        algoritmo,
        tipo,
        tamanho,
        repeticao,
        resultado["tempo_s"],
        resultado["pico_memoria_bytes"],
        resultado["estados"],
        resultado["existe"],
        False,
    )

### Instâncias Backtracking vs Meet In Middle

In [6]:
tarefas = []
for tipo, tamanho in CENARIOS:
    instancias = gerador.gerar_lote(tamanho, REPETICOES, tipo, SEMENTE_BASE)

    for algoritmo in ("backtracking", "meet_in_middle"):
        for repeticao, instancia in enumerate(instancias):
            tarefas.append((tipo, tamanho, algoritmo, repeticao, instancia, True))

medicoes_backtracking_vs_mim = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(medir, *t): t for t in tarefas}
    for future in as_completed(futures):
        medicoes_backtracking_vs_mim.append(future.result())

### Instâncias Backtracking com Poda vs Sem Poda:

In [7]:
tarefas = []
for tamanho in TAMANHOS:
    instancias = gerador.gerar_lote(tamanho, REPETICOES, "com_solucao", SEMENTE_BASE)

    for usar_poda in (True, False):
        for repeticao, instancia in enumerate(instancias):
            tarefas.append(("com_solucao", tamanho, "backtracking", repeticao, instancia, usar_poda))

medicoes_backtracking_com_vs_sem_poda = []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(medir, *t): t for t in tarefas}
    for future in as_completed(futures):
        medicoes_backtracking_com_vs_sem_poda.append(future.result())

### Exportar Dados em CSV

In [8]:
salvar_csv(medicoes_backtracking_vs_mim, RESULTADOS_BACKTRACKING_VS_MIM_PATH)
salvar_csv(medicoes_backtracking_com_vs_sem_poda, RESULTADOS_BACKTRACKING_EFICIENCIA_PODA)